# Imports

In [1]:
import importlib
import sys
import torch

sys.path.insert(0, '../..')
sys.path.insert(0, '../../../')
sys.path.insert(0, '../../../../../load/event_log_loader')

import new_event_log_loader

# Data

### Load Data Files

In [2]:
# Path to your pickle file (saved with torch.save)
file_path_train = '../../../../../load/encoded_data/helpdesk_all_1_train.pkl'
# Load the dataset using torch.load
helpdesk_train_dataset = torch.load(file_path_train, weights_only=False)
# Check the type of the loaded dataset
print(type(helpdesk_train_dataset))

# Path to your pickle file (saved with torch.save)
file_path_val = '../../../../../load/encoded_data/helpdesk_all_1_val.pkl'
# Load the dataset using torch.load
helpdesk_val_dataset = torch.load(file_path_val, weights_only=False)
# Check the type of the loaded dataset
print(type(helpdesk_val_dataset))


<class 'new_event_log_loader.EventLogDataset'>
<class 'new_event_log_loader.EventLogDataset'>


### Train Data Insights

In [3]:
# Helpdesk Dataset Categories, Features:
helpdesk_all_categories = helpdesk_train_dataset.all_categories

helpdesk_all_categories_cat = helpdesk_all_categories[0]
print(helpdesk_all_categories_cat)

helpdesk_all_categories_num = helpdesk_all_categories[1]
print(helpdesk_all_categories_num)

for i, cat in enumerate(helpdesk_all_categories_cat):
     print(f"Helpdesk (5) Categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk (5) Total Amount of Category labels: {cat[1]}")

print('\n')    

for i, num in enumerate(helpdesk_all_categories_num):
     print(f"Helpdesk (5) Numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk (5) Amount Numerical: {num[1]}")
     
# Get concept_name id:
# 
concept_name = 'Activity_start'
concept_name_id = [i for i, cat in enumerate(helpdesk_all_categories[0]) if cat[0] == concept_name][0]

print("ID concet name in cat list: ", concept_name_id)

duration_seconds = 'duration_seconds'
duration_seconds_id = [i for i, num in enumerate(helpdesk_all_categories[1]) if num[0] == duration_seconds][0]
print("ID duration_seconds in num list: ", duration_seconds_id)

[('Activity_start', 13, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'Insert ticket': 5, 'Require upgrade': 6, 'Resolve SW anomaly': 7, 'Resolve ticket': 8, 'Schedule intervention': 9, 'Take in charge ticket': 10, 'VERIFIED': 11, 'Wait': 12}), ('Resource_start', 23, {'Value 1': 1, 'Value 10': 2, 'Value 11': 3, 'Value 12': 4, 'Value 13': 5, 'Value 14': 6, 'Value 15': 7, 'Value 16': 8, 'Value 17': 9, 'Value 18': 10, 'Value 19': 11, 'Value 2': 12, 'Value 20': 13, 'Value 21': 14, 'Value 22': 15, 'Value 3': 16, 'Value 4': 17, 'Value 5': 18, 'Value 6': 19, 'Value 7': 20, 'Value 8': 21, 'Value 9': 22})]
[('seconds_in_day', 1, {}), ('day_in_week', 1, {}), ('duration_seconds', 1, {})]
Helpdesk (5) Categorical feature: Activity_start, Index position in categorical data list: 0
Helpdesk (5) Total Amount of Category labels: 13
Helpdesk (5) Categorical feature: Resource_start, Index position in categorical data list: 1
Helpdesk (5) Total Amount of Category labels: 

In [4]:
selected_cat_attributes = ['Activity_start', 'Resource_start']
selected_num_attributes = ['seconds_in_day', 'day_in_week']

selected_categories = (
    [cat for cat in helpdesk_all_categories[0] if cat[0] in selected_cat_attributes],
    [num for num in helpdesk_all_categories[1] if num[0] in selected_num_attributes]
)

# Loss Object Creation

# Training Configuration

In [5]:
import stochasticLSTM.model

importlib.reload(stochasticLSTM.model)
from stochasticLSTM.model import StochasticLSTM

"""
Specific model parameters from paper: 
"""

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")

# Size hidden layer
hidden_size = 128

# Number of LSTM cells
num_layers = 2

# Fixed Dropout probability
p_fix = 0.1

# Lambda for L2 (weight, bias, dropout) regularization: According to formula: 1/2N
regularization_term = 1e-5

# Hans Weytjens LSTM model
model = StochasticLSTM(
    data_set_categories=helpdesk_all_categories,
    model_input_feat=selected_categories,
    hidden_size=hidden_size,
    num_layers=num_layers,
    weight_reg=regularization_term,
    p_fix=p_fix,
    device=device,
)

import loss.losses

importlib.reload(loss.losses)
from loss.losses import Loss

loss_obj = Loss()


import training.train

importlib.reload(training.train)
from training.train import Training

from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(comment="train")


"""
Parameter of Probabilistic Suffix Prediction experimental design, to ensure fair comparison:
"""

# Start learning rate
learning_rate = 5e-3

# Optimizer and Scheduler
optimizer = torch.optim.Adam(
    params=model.parameters(), lr=learning_rate, weight_decay=0
)
scheduler = ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=20, min_lr=1e-10
)

# Epochs
num_epochs = 200

# Batch of model input
batch_size = 128

# shuffle data
shuffle = True

optimize_values = {
    "optimizer": optimizer,
    "scheduler": scheduler,
    "epochs": num_epochs,
    "mini_batches": batch_size,
    "shuffle": shuffle,
}

trainer = Training(
    model=model,
    device=device,
    data_train=helpdesk_train_dataset,
    data_val=helpdesk_val_dataset,
    selected_features=(selected_cat_attributes, selected_num_attributes),
    concept_name_id=concept_name_id,
    duration_seconds_id=duration_seconds_id,
    loss_obj=loss_obj,
    optimize_values=optimize_values,
    writer=writer,
    save_model_n_th_epoch=1,
    saving_path="model.pkl",
)

# Train the model:
trainer.train()

Embeddings:  ModuleList(
  (0): Embedding(13, 16)
  (1): Embedding(23, 16)
)
Total embedding feature size:  32
Input feature size:  34
Cells hidden size:  128
Number of LSTM layer:  2
Dropout rate:  0.1


Device:  cuda
Optimizer:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0
)
Scheduler:  <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x7fc89b846260>
Epochs:  200
Mini baches:  128
Shuffle batched dataset:  True


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/90 [00:00<?, ?it/s]

Epoch [1/200], Learning Rate: 0.005
Training: Avg Attenuated Training Loss: 0.2610
Validation: Avg Standard Validation Loss: 0.4203
Validation: Avg Attenuated Validation Loss: -0.0323
Validation Loss for Scheduler: 0.4203
saving model


  0%|          | 0/90 [00:00<?, ?it/s]

...

Epoch [198/200], Learning Rate: 7.8125e-05
Training: Avg Attenuated Training Loss: -0.1758
Validation: Avg Standard Validation Loss: 0.3778
Validation: Avg Attenuated Validation Loss: 11.1268
Validation Loss for Scheduler: 0.3778
saving model


  0%|          | 0/90 [00:00<?, ?it/s]

Epoch [199/200], Learning Rate: 7.8125e-05
Training: Avg Attenuated Training Loss: -0.2607
Validation: Avg Standard Validation Loss: 0.3869
Validation: Avg Attenuated Validation Loss: 12.1143
Validation Loss for Scheduler: 0.3869
saving model


  0%|          | 0/90 [00:00<?, ?it/s]

Epoch [200/200], Learning Rate: 7.8125e-05
Training: Avg Attenuated Training Loss: -0.1317
Validation: Avg Standard Validation Loss: 0.3870
Validation: Avg Attenuated Validation Loss: 19.8056
Validation Loss for Scheduler: 0.3870
saving model
Training complete.
Model saved to path: model.pkl
